# 1 · Data ingestion, ETL & SQL | 45 minutes
**Business mission:** clean a small sample of real Kaggle customer-support tickets and load them into your **team-owned Supabase tables**. Then answer business questions with SQL.

Source: [Customer Support Ticket Dataset (Kaggle)](https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset). **Instructor already downloaded + ingested it**; no Kaggle account or local CSV required.

> A source caveat: `First Response Time` and `Time to Resolution` are timestamps. Their difference is **time from first response to resolution**, not total ticket lifetime or a contractual SLA. Missing durations stay null. Dataset is synthetic and may have noisy text.

In [ ]:
%pip install -q "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "plotly>=5,<7" "openai>=1.55,<3"

## Connect (provided helper; only your team tables are writable)

In [ ]:
import os, getpass
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

## Explore schema before changing data
**Exercise:** Name one primary key, one foreign key, and one reason the team has its own schema. The instructor has already created the tables—do not run DDL.

In [ ]:
display(fetch_df("""SELECT table_schema, table_name, column_name, data_type
  FROM information_schema.columns
  WHERE (table_schema='shared' AND table_name IN ('raw_tickets','ticket_types','knowledge_articles'))
     OR (table_schema=%s AND table_name IN ('clean_tickets','knowledge_articles'))
  ORDER BY table_schema, table_name, ordinal_position""",(TEAM_SCHEMA,)).head(45))

In [ ]:
raw = fetch_df("SELECT * FROM shared.raw_tickets ORDER BY ticket_id LIMIT 600")
print("Raw records:",len(raw)); display(raw.head(3))
print("Null counts:"); display(raw.isna().sum().to_frame('nulls').T)

## TODO 1 · Clean the tickets
Implement **three targeted expressions**; leave the rest unchanged. Filter rows with missing ticket identifiers or ticket types, normalize string fields, and deduplicate. Never impute a missing resolution timestamp as zero hours.

In [ ]:
df = raw.copy()
df["ticket_id"] = pd.to_numeric(df["ticket_id"], errors="coerce")
df["ticket_type"] = df["ticket_type"].astype("string").str.strip()
df["status"] = df["status"].astype("string").str.strip().str.title()

# TODO 1: Remove rows with missing ticket ID or ticket type
df = df.dropna(subset=[________, ________])
# TODO 2: Keep one row for each ticket ID
df = df.drop_duplicates(subset=[________], keep="last")

for col in ("first_response_at", "resolved_at"):
    df[col] = pd.to_datetime(df[col],utc=True,errors="coerce")
df["satisfaction_rating"] = pd.to_numeric(df["satisfaction_rating"],errors="coerce")

# TODO 3: Compute hours from first response to resolution (not total ticket lifetime)
hours = (df[________] - df[________]).dt.total_seconds() / 3600
df["response_to_resolution_hours"] = hours.where(
    df["status"].eq("Closed") & hours.between(0, 24*365)
).round(2)

df = df.astype(object).where(pd.notna(df), None)
print("Cleaned tickets:",len(df))
display(df[["ticket_id","ticket_type","status","response_to_resolution_hours"]].head())

## Load into your team table — prewritten UPSERT

In [ ]:
# Provided loader. Idempotent UPSERT makes reruns safe.
from datetime import datetime
columns = ["ticket_id","ticket_type","ticket_subject","status","priority","channel", "product",
           "first_response_at","resolved_at","satisfaction_rating","response_to_resolution_hours"]
allowed_types = set(fetch_df("SELECT ticket_type FROM shared.ticket_types")["ticket_type"])
ready = df[columns].copy()
ready = ready[ready["ticket_type"].isin(allowed_types)]
ready["ticket_id"] = ready["ticket_id"].astype(int)

def pgval(v):
    if v is None or (not isinstance(v, (list,tuple,dict)) and pd.isna(v)): return None
    if hasattr(v,"to_pydatetime"): return v.to_pydatetime()
    return v

placeholders=','.join(['%s']*len(columns))
updates=','.join(f"{c}=EXCLUDED.{c}" for c in columns if c != "ticket_id")
query=f"""INSERT INTO {TEAM_SCHEMA}.clean_tickets ({','.join(columns)})
VALUES ({placeholders}) ON CONFLICT(ticket_id) DO UPDATE SET {updates}"""
with closing(connect()) as conn:
    with conn.transaction():
        with conn.cursor() as cur:
            cur.executemany(query,[tuple(pgval(v) for v in row) for row in ready.itertuples(index=False,name=None)])
print("Loaded:",len(ready),"team rows")
assert int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0,0]) >= 1

## TODO 4 · Business SQL: JOIN and aggregation

In [ ]:
# TODO 4: Complete JOIN and GROUP BY (no instructor DDL needed).
q = f"""
SELECT t.ticket_type, COUNT(*) AS total_tickets,
       ROUND(AVG(t.response_to_resolution_hours), 2) AS avg_response_to_resolution_hours
FROM {TEAM_SCHEMA}.clean_tickets t
JOIN shared.ticket_types c ON __________
GROUP BY __________
ORDER BY total_tickets DESC
"""
display(fetch_df(q))

## TODO 5 · Closed rate (a business metric, not official SLA)

In [ ]:
# TODO 5: Use FILTER to count closed tickets and calculate closed rate.
q = f"""
SELECT ticket_type,
       COUNT(*) AS total,
       COUNT(*) FILTER (WHERE status = '________') AS closed,
       ROUND(100.0 * COUNT(*) FILTER (WHERE status = '________') / NULLIF(COUNT(*),0), 1) AS closed_pct
FROM {TEAM_SCHEMA}.clean_tickets
GROUP BY ticket_type ORDER BY total DESC
"""
display(fetch_df(q))

### Checkpoint
You should see results for each ticket type, and the team-owned table should contain cleaned rows. If something fails, ask the instructor to share a solution cell and re-run—the subsequent notebooks read this same table.

In [ ]:
# Recovery checkpoint: use this ONLY if your ETL was blocked.
# The instructor can run the same cleaning logic and load a reference snapshot.
RECOVER_FROM_CHECKPOINT = False
if RECOVER_FROM_CHECKPOINT:
    print("Ask the instructor to run the reference solution before continuing.")

print("Verified rows:", int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0,0]))